In [0]:
grren_taxi_df = spark.readStream.table("NYCTAXI.BRONZE.GREEN_TAXI")

silver_df = (
    grren_taxi_df
    # Example: rename columns
    .withColumnRenamed("VendorID", "vendor_id")
    .withColumnRenamed("passenger_count", "passengers")
    # Example: cast types
    .withColumn("trip_distance", col("trip_distance").cast("double"))
    .withColumn("pickup_datetime", col("lpep_pickup_datetime").cast("timestamp"))
    .withColumn("dropoff_datetime", col("lpep_dropoff_datetime").cast("timestamp"))
    # Keep metadata for lineage
    .withColumn("file_name", col("file_name"))
    .withColumn("file_path", col("file_path"))
    .withColumn("load_timestamp", col("load_timestamp"))
)

(silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/nyctaxi/checkpoints/silver/green_taxi")
    .table("NYCTAXI.SILVER.GREEN_TAXI"))
